<a href="https://colab.research.google.com/github/Padmadharshini03/Machine-learning-/blob/main/experiment_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Program 4: Artificial Neural Network with the Backpropagation algorithm
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))


def sigmoid_derivative(a):
    # a is already sigmoid(x)
    return a * (1.0 - a)


class NeuralNetwork:
    """Input layer -> one hidden layer -> output layer (sigmoid activations)."""

    def __init__(self, n_input, n_hidden, n_output, lr=0.5, seed=1):
        rng = np.random.default_rng(seed)
        self.W1 = rng.uniform(-1, 1, (n_input, n_hidden))
        self.b1 = np.zeros((1, n_hidden))
        self.W2 = rng.uniform(-1, 1, (n_hidden, n_output))
        self.b2 = np.zeros((1, n_output))
        self.lr = lr

    def forward(self, X):
        self.a1 = sigmoid(X @ self.W1 + self.b1)          # hidden layer output
        self.a2 = sigmoid(self.a1 @ self.W2 + self.b2)    # network output
        return self.a2

    def backward(self, X, y):
        m = X.shape[0]
        # error terms (deltas)
        delta_out = (y - self.a2) * sigmoid_derivative(self.a2)
        delta_hidden = (delta_out @ self.W2.T) * sigmoid_derivative(self.a1)
        # weight updates
        self.W2 += self.lr * (self.a1.T @ delta_out) / m
        self.b2 += self.lr * delta_out.sum(axis=0, keepdims=True) / m
        self.W1 += self.lr * (X.T @ delta_hidden) / m
        self.b1 += self.lr * delta_hidden.sum(axis=0, keepdims=True) / m

    def train(self, X, y, epochs=10000, print_every=1000):
        for epoch in range(1, epochs + 1):
            output = self.forward(X)
            self.backward(X, y)
            if epoch % print_every == 0:
                loss = np.mean((y - output) ** 2)
                print(f"Epoch {epoch:6d}   MSE loss = {loss:.6f}")

    def predict(self, X):
        return self.forward(X)


# ---------------------------------------------------------------
# Test 1: XOR problem
# ---------------------------------------------------------------
print("=== Test 1: XOR problem ===")
X_xor = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=float)
y_xor = np.array([[0], [1], [1], [0]], dtype=float)

nn = NeuralNetwork(n_input=2, n_hidden=4, n_output=1, lr=2.0, seed=1)
nn.train(X_xor, y_xor, epochs=20000, print_every=4000)

print("\nInput  Target  Predicted  Rounded")
for xi, ti, pi in zip(X_xor, y_xor, nn.predict(X_xor)):
    print(f"{xi}   {int(ti[0])}      {pi[0]:.4f}     {int(round(pi[0]))}")

# ---------------------------------------------------------------
# Test 2: Iris data set (3 classes)
# ---------------------------------------------------------------
print("\n=== Test 2: Iris data set ===")
iris = load_iris()
X = iris.data
y = iris.target

# one-hot encoding of the class labels
Y = np.eye(3)[y]

# scale the features
X = StandardScaler().fit_transform(X)

X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=0.2, random_state=42, stratify=y)

nn2 = NeuralNetwork(n_input=4, n_hidden=8, n_output=3, lr=1.0, seed=1)
nn2.train(X_train, Y_train, epochs=5000, print_every=1000)

pred = np.argmax(nn2.predict(X_test), axis=1)
actual = np.argmax(Y_test, axis=1)
accuracy = np.mean(pred == actual) * 100
print(f"\nTest accuracy on Iris: {accuracy:.2f}%")
print("Predicted :", pred)
print("Actual    :", actual)
print("END")

=== Test 1: XOR problem ===
Epoch   4000   MSE loss = 0.000760
Epoch   8000   MSE loss = 0.000313
Epoch  12000   MSE loss = 0.000195
Epoch  16000   MSE loss = 0.000140
Epoch  20000   MSE loss = 0.000110

Input  Target  Predicted  Rounded
[0. 0.]   0      0.0069     0
[0. 1.]   1      0.9899     1
[1. 0.]   1      0.9890     1
[1. 1.]   0      0.0130     0

=== Test 2: Iris data set ===
Epoch   1000   MSE loss = 0.015003
Epoch   2000   MSE loss = 0.011046
Epoch   3000   MSE loss = 0.009870
Epoch   4000   MSE loss = 0.009274
Epoch   5000   MSE loss = 0.008909

Test accuracy on Iris: 100.00%
Predicted : [0 2 1 1 0 1 0 0 2 1 2 2 2 1 0 0 0 1 1 2 0 2 1 2 2 1 1 0 2 0]
Actual    : [0 2 1 1 0 1 0 0 2 1 2 2 2 1 0 0 0 1 1 2 0 2 1 2 2 1 1 0 2 0]
END
